# Snowflake Optima Tutorial

This notebook explains **Snowflake Optima** - an intelligent, automatic query optimization feature that improves performance without manual tuning.

## Documentation Links
- [Snowflake Optima Overview](https://docs.snowflake.com/en/user-guide/snowflake-optima)
- [Search Optimization Service](https://docs.snowflake.com/en/user-guide/search-optimization-service) (underlying technology)
- [Query Profile in Snowsight](https://docs.snowflake.com/en/user-guide/ui-query-profile)
- [Optimizing Query Performance](https://docs.snowflake.com/en/user-guide/performance-query-options)

---

## What is Snowflake Optima?

Snowflake Optima **automatically analyzes your workload patterns** and implements optimization strategies without requiring manual configuration. It continuously monitors your queries and creates hidden indexes to accelerate repetitive workloads.

### Key Benefits:
- **Zero configuration** - works automatically in the background
- **No additional cost** - included with Gen2 warehouses
- **Intelligent indexing** - learns from your query patterns
- **Automatic maintenance** - indexes are created and updated automatically

---

## Requirements

### Generation 2 Warehouses Required

> **Important:** Snowflake Optima is **only available on Generation 2 (Gen2) standard warehouses**.

Gen2 warehouses are the default for new accounts. To check if your warehouse is Gen2, look for `WAREHOUSE_TYPE = 'STANDARD'` in SHOW WAREHOUSES output.

### Check Your Warehouse Type

In [ ]:
%%sql -r warehouse_info
SHOW WAREHOUSES LIKE 'COMPUTE_WH';

---

## How Optima Indexing Works

### The Process:

1. **Workload Analysis** - Optima continuously monitors your SQL queries
2. **Pattern Detection** - Identifies repetitive point-lookup queries and filter patterns
3. **Index Creation** - Automatically creates hidden indexes on relevant columns
4. **Partition Pruning** - Uses indexes to skip irrelevant micro-partitions during query execution

### What Optima Optimizes:
- **Point lookups** - `WHERE id = 12345`
- **Equality filters** - `WHERE status = 'ACTIVE'`
- **IN clauses** - `WHERE region IN ('US-East', 'US-West')`
- **Repetitive query patterns** - Queries run frequently against the same tables

### Built on Search Optimization
Optima Indexing is built on top of the [Search Optimization Service](https://docs.snowflake.com/en/user-guide/search-optimization-service). The difference is:

| Feature | Optima Indexing | Search Optimization |
|---------|-----------------|--------------------|
| Configuration | Automatic | Manual (ALTER TABLE) |
| Cost | Free (Gen2 only) | Storage + Compute costs |
| Freshness | Best-effort | Guaranteed |
| Use case | General workloads | Critical/real-time workloads |

---

## Monitoring Optima Benefits

### Method 1: Query Profile in Snowsight (UI)

After running a query, check the **Query Profile** tab in Snowsight:

1. **Query Insights Pane** - Look for "Snowflake Optima used" indicator
2. **Statistics Pane** - Check "Partitions pruned by Snowflake Optima" metric

### Method 2: Query via ACCOUNT_USAGE Views

The following queries help identify queries benefiting from Optima. Note that Optima-specific columns are visible in the Query Profile UI, but we can infer benefit from pruning statistics.

---

## Identifying High-Pruning Queries (Optima Candidates)

Queries with high partition pruning ratios are likely benefiting from Optima or other optimizations:

In [ ]:
%%sql -r high_pruning_queries
USE ROLE ACCOUNTADMIN;

SELECT 
    query_id,
    query_text,
    warehouse_name,
    total_elapsed_time / 1000 AS elapsed_sec,
    partitions_scanned,
    partitions_total,
    ROUND((partitions_total - partitions_scanned) / NULLIF(partitions_total, 0) * 100, 1) AS pruning_pct,
    bytes_scanned / (1024*1024) AS mb_scanned,
    start_time
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
  AND partitions_total > 100
  AND execution_status = 'SUCCESS'
  AND query_type = 'SELECT'
ORDER BY pruning_pct DESC NULLS LAST
LIMIT 20;

---

## Analyzing Point Lookup Queries

Point lookups (equality filters on single values) are prime candidates for Optima optimization:

In [ ]:
%%sql -r point_lookups
SELECT 
    query_parameterized_hash,
    ANY_VALUE(SUBSTRING(query_text, 1, 200)) AS sample_query,
    COUNT(*) AS execution_count,
    ROUND(AVG(total_elapsed_time), 0) AS avg_elapsed_ms,
    ROUND(AVG(partitions_scanned), 0) AS avg_partitions_scanned,
    ROUND(AVG(partitions_total), 0) AS avg_partitions_total,
    ROUND(AVG((partitions_total - partitions_scanned) / NULLIF(partitions_total, 0) * 100), 1) AS avg_pruning_pct
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND query_type = 'SELECT'
  AND partitions_total > 0
  AND (LOWER(query_text) LIKE '%where%=%' OR LOWER(query_text) LIKE '%where% in %')
GROUP BY query_parameterized_hash
HAVING COUNT(*) >= 5
ORDER BY execution_count DESC
LIMIT 15;

---

## Tables Most Likely Benefiting from Optima

Tables with many queries and good pruning ratios likely have Optima indexes:

In [ ]:
%%sql -r optima_tables
SELECT 
    SPLIT_PART(query_text, 'FROM ', 2) AS table_hint,
    COUNT(*) AS query_count,
    ROUND(AVG(total_elapsed_time), 0) AS avg_elapsed_ms,
    ROUND(AVG(partitions_scanned), 0) AS avg_partitions_scanned,
    ROUND(AVG(partitions_total), 0) AS avg_partitions_total,
    ROUND(AVG((partitions_total - partitions_scanned) / NULLIF(partitions_total, 0) * 100), 1) AS avg_pruning_pct,
    SUM(bytes_scanned) / (1024*1024*1024) AS total_gb_scanned
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND query_type = 'SELECT'
  AND partitions_total > 100
  AND UPPER(query_text) LIKE 'SELECT%FROM%WHERE%'
GROUP BY table_hint
HAVING COUNT(*) >= 10 AND avg_pruning_pct > 50
ORDER BY query_count DESC
LIMIT 15;

---

## Query Performance Trends

Track if similar queries are improving over time (Optima learns and adapts):

In [ ]:
%%sql -r daily_trends
SELECT 
    DATE_TRUNC('day', start_time) AS query_date,
    COUNT(*) AS total_queries,
    ROUND(AVG(total_elapsed_time), 0) AS avg_elapsed_ms,
    ROUND(AVG(partitions_scanned), 0) AS avg_partitions_scanned,
    ROUND(AVG((partitions_total - partitions_scanned) / NULLIF(partitions_total, 0) * 100), 1) AS avg_pruning_pct,
    SUM(bytes_scanned) / (1024*1024*1024) AS total_gb_scanned
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD('day', -14, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND query_type = 'SELECT'
  AND partitions_total > 0
GROUP BY query_date
ORDER BY query_date;

---

## Comparing Search Optimization vs Optima

If you have tables with explicit Search Optimization enabled, compare their performance:

In [ ]:
%%sql -r search_opt_tables
-- Show tables with search optimization enabled
SELECT 
    table_catalog AS database_name,
    table_schema AS schema_name, 
    table_name,
    search_optimization,
    search_optimization_progress,
    search_optimization_bytes
FROM SNOWFLAKE.ACCOUNT_USAGE.TABLE_STORAGE_METRICS
WHERE search_optimization = 'ON'
  AND DELETED IS NULL
ORDER BY search_optimization_bytes DESC
LIMIT 20;

---

## Viewing Search Optimization Benefits

For tables with **explicit** search optimization (not Optima), use this view:

In [ ]:
%%sql -r search_benefits
-- Query search optimization benefits (requires ACCOUNTADMIN)
-- This view shows partitions pruned due to search optimization
SELECT *
FROM SNOWFLAKE.ACCOUNT_USAGE.SEARCH_OPTIMIZATION_BENEFITS
WHERE end_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY partitions_pruned_additional DESC
LIMIT 20;

---

## How to Verify Optima is Working (Per Query)

To confirm Optima helped a specific query:

### Step 1: Run your query and note the Query ID
### Step 2: View Query Profile in Snowsight

1. Go to **Query History** in Snowsight
2. Find your query and click on it
3. Click the **Query Profile** tab
4. Look at the **Statistics** pane for:
   - `Partitions pruned by Snowflake Optima`
5. Check **Query Insights** pane for:
   - "Snowflake Optima used" indicator

### Example Query to Test

In [ ]:
%%sql -r test_instructions
-- Example: Run a point lookup query to test Optima
-- Replace with your own table and filter column

-- SELECT * 
-- FROM your_database.your_schema.your_large_table
-- WHERE your_id_column = 12345;

-- After running, check Query Profile in Snowsight for:
-- "Partitions pruned by Snowflake Optima" in Statistics pane

SELECT 'Run a point lookup on your data, then check Query Profile in Snowsight' AS instructions;

---

## Best Practices for Optima

### Let Optima Work for You:
1. **Use Gen2 warehouses** - Optima only works on Gen2 standard warehouses
2. **Run queries consistently** - Optima learns from repetitive patterns
3. **Use specific filters** - Point lookups benefit most
4. **Be patient** - Index creation happens in the background over time

### When to Use Explicit Search Optimization Instead:

| Use Optima When... | Use Search Optimization When... |
|-------------------|--------------------------------|
| General workloads | Mission-critical latency requirements |
| Budget-conscious | Guaranteed index freshness needed |
| Varied query patterns | Specific tables need guaranteed optimization |
| No configuration desired | Willing to pay for storage/compute |

### Configure Search Optimization (if needed):
```sql
-- Enable search optimization on a specific table
ALTER TABLE my_table ADD SEARCH OPTIMIZATION;

-- Enable on specific columns only
ALTER TABLE my_table ADD SEARCH OPTIMIZATION 
  ON EQUALITY(column1, column2);

-- Check configuration
DESCRIBE SEARCH OPTIMIZATION ON my_table;
```

---

## Summary

### Key Takeaways:

1. **Automatic** - Optima requires no configuration; it just works on Gen2 warehouses
2. **Free** - No additional cost beyond your Gen2 warehouse compute
3. **Intelligent** - Learns from your workload patterns over time
4. **Monitor via UI** - Check Query Profile for "Partitions pruned by Snowflake Optima"
5. **Best for point lookups** - Equality filters and IN clauses benefit most

### Monitoring Checklist:
- ✅ Check Query Profile → Statistics → "Partitions pruned by Snowflake Optima"
- ✅ Check Query Profile → Query Insights → "Snowflake Optima used"
- ✅ Track pruning percentages over time to see Optima learning
- ✅ For critical workloads, consider explicit Search Optimization

### Documentation Reference:
- [Snowflake Optima](https://docs.snowflake.com/en/user-guide/snowflake-optima)
- [Search Optimization Service](https://docs.snowflake.com/en/user-guide/search-optimization-service)
- [Query Profile](https://docs.snowflake.com/en/user-guide/ui-query-profile)